# Notebook 5 — Model Evaluation & Bias Analysis
### AIMI High School Internship

You've trained a pneumonia classifier. Training is only half the story: a model that looks great *on average* can quietly **fail specific groups of patients** — and in medicine that can mean missed diagnoses for exactly the people a screening tool is supposed to protect.

> In 2019, a widely-used healthcare algorithm was found to systematically underserve Black patients — not by design, but because of bias baked into its data. Hidden bias like this is one of the biggest challenges in medical AI.

**Think about it:** if your model is 90% accurate overall but only 70% accurate for elderly patients, should a hospital deploy it? In this notebook you'll **measure performance deeply, audit for bias across subgroups, formalize what "fair" even means, look inside the model with SHAP, and try fixes.**

## What you'll do in this notebook

1. **Setup & predictions** — load a model and run it on the validation set.
2. **Patient metadata** — pull age / sex / view position for every patient.
3. **Overall performance** — accuracy, precision/recall, ROC — beyond just accuracy.
4. **Bias by subgroup** — does the model work equally well across sex, age, and view?
5. **Formalizing fairness** — independence vs. separation vs. sufficiency (the COMPAS debate).
6. **Where it fails** — inspect the actual misclassified X-rays.
7. **Looking inside — SHAP** — which pixels drove each prediction?
8. **Fixes** — data augmentation and per-group thresholds.
9. **Ethics reflection**.

# 1. Setup & predictions

> Before running: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
#@title Notebook settings { display-mode: "form" }
# On Colab, leave these as-is and just run every cell.
# (Automated validation overrides DATA_DIR via papermill — do not rename this cell.)
from pathlib import Path
# Put `dataset_1.zip` in your Google Drive ("My Drive"), then run all cells.
DATA_DIR = Path("/content/drive/MyDrive/dataset_1")

EPOCHS = 3  #@param {type:"integer"}
RUN_SHAP = True  #@param {type:"boolean"}

In [ ]:
#@title Run setup (imports · data · device) { display-mode: "form" }
import os, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve)
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.transforms import functional as TF
from torch.utils.data import Dataset, DataLoader

# Mount Google Drive so we can read the dataset (skipped automatically off Colab).
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

# Unzip the dataset to Colab's local disk and read from there. Reading images over the Drive
# mount re-fetches each file from the network on every read (slow); we unzip the dataset's
# .zip once to /content instead. Students put dataset_1.zip / dataset_2.zip in their Drive
# (My Drive). Skipped when DATA_DIR already points at local data (validation / off-Colab).
import zipfile
DATA_DIR = Path(DATA_DIR)
if str(DATA_DIR).startswith("/content/drive/"):
    local = Path("/content") / DATA_DIR.name
    if not local.exists():
        zip_path = Path("/content/drive/MyDrive") / f"{DATA_DIR.name}.zip"
        assert zip_path.exists(), (
            f"{zip_path} not found — add {DATA_DIR.name}.zip to your Google Drive (My Drive).")
        print(f"Unzipping {DATA_DIR.name}.zip to local disk (one-time)...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(local)                            # everything lands under /content/<name>
        kids = [p for p in local.iterdir() if not p.name.startswith((".", "__"))]
        if len(kids) == 1 and kids[0].is_dir():            # zip wrapped files in one extra folder
            import shutil
            for it in list(kids[0].iterdir()):
                shutil.move(str(it), str(local / it.name))
            kids[0].rmdir()
    DATA_DIR = local
assert DATA_DIR.exists(), f"dataset not found at {DATA_DIR}."
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_DIR, VAL_DIR = DATA_DIR / "train_images", DATA_DIR / "val_images"

train_labels = pd.read_csv(DATA_DIR / "train_labels.csv")
val_labels   = pd.read_csv(DATA_DIR / "val_labels.csv")
print(f"device={DEVICE} | val patients={val_labels['patientId'].nunique()}")

## 1.1 Dataset & your model
One row per patient (a patient's box rows collapse to a single `Target`).

A bias audit is only meaningful on a model you'd actually trust — so don't audit a throwaway baseline. **Bring your best classifier from Notebook 2** (the transfer-learned ResNet, with whatever backbone / augmentation / learning rate scored highest for you) and train it below — that is the model whose fairness we'll audit.

> **Why Notebook 2 and not Notebook 3?** Everything here — subgroup accuracy / AUROC, the confusion-matrix fairness criteria, threshold tuning, SHAP — evaluates a **binary pneumonia classifier**. Notebook 3 is *detection* (bounding boxes), which doesn't map onto these per-patient classification metrics. So the model under audit is your **classifier** from Notebook 2.

**One interface rule:** the prediction cell reads `softmax(model(x))[:, 1]` as `P(pneumonia)`, so your model must output **2 logits** per image — use a 2-class head (`nn.Linear(in_features, 2)`) even if you used a single-sigmoid head in Notebook 2.

In [ ]:
class Dataset1(Dataset):
    def __init__(self, df, images_dir, transform=None):
        self.df = df.groupby("patientId", as_index=False)["Target"].max()    # one row/patient
        self.images_dir, self.transform = Path(images_dir), transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.images_dir / f"{row['patientId']}.jpg").convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, int(row["Target"]), row["patientId"]

val_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                             transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
val_dataset = Dataset1(val_labels, VAL_DIR, transform=val_tf)
val_loader  = DataLoader(val_dataset, batch_size=32, shuffle=False)

train_tf = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(),
                               transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
train_loader = DataLoader(Dataset1(train_labels, TRAIN_DIR, transform=train_tf), batch_size=32, shuffle=True)

# Build + train YOUR best classifier from Notebook 2. It must end as a trained `model`
# (in eval mode) that outputs 2 logits per image, so the prediction cell below works.
# TODO: implement YOUR best classifier from Notebook 2.
#   1. start from a PRETRAINED backbone (transfer learning beats from-scratch):
#        model = models.resnet18(weights="DEFAULT")     # or your best NB2 backbone
#   2. swap in a fresh 2-CLASS head so the eval pipeline's softmax(...)[:, 1] works:
#        model.fc = nn.Linear(model.fc.in_features, 2)
#      (use 2 outputs here even if you used a single-sigmoid head in Notebook 2)
#   3. model = model.to(DEVICE)
#   4. train it: class-weighted nn.CrossEntropyLoss + Adam(lr=1e-4), looping EPOCHS over
#      train_loader  (each batch unpacks as  images, labels, _)
#   5. model.eval()
model = ...  # YOUR CODE HERE
raise NotImplementedError("Implement your best model from Notebook 2, then delete this line.")

## 1.2 Generate predictions
Run the model over the validation set and store, for each patient, the true label, the predicted label, and `P(pneumonia)`.

In [ ]:
all_pids, all_labels, all_preds, all_probs = [], [], [], []
model.eval()
with torch.no_grad():
    for images, labels, pids in tqdm(val_loader, desc="inference"):
        probs = torch.softmax(model(images.to(DEVICE)), dim=1)[:, 1].cpu().numpy()
        all_pids.extend(pids); all_labels.extend(labels.numpy())
        all_preds.extend((probs >= 0.5).astype(int)); all_probs.extend(probs)

results_df = pd.DataFrame({"patientId": all_pids, "true_label": all_labels,
                           "predicted": all_preds, "prob_pneumonia": all_probs})
print(f"predictions for {len(results_df)} patients"); results_df.head()

# 2. Patient metadata

For bias analysis we need each patient's **age, sex, and view position**. In this dataset those ship right in the labels CSV (they were pulled from the original DICOM headers), so we just merge them in — no DICOM parsing required.

> *In a raw clinical pipeline these live in the DICOM header; here they're columns.*

In [ ]:
meta_df = (val_labels.drop_duplicates("patientId")
           [["patientId", "PatientAge", "PatientSex", "ViewPosition"]]
           .rename(columns={"PatientAge": "age", "PatientSex": "sex", "ViewPosition": "view_position"}))
meta_df["age"] = pd.to_numeric(meta_df["age"], errors="coerce")
meta_df["age_group"] = pd.cut(meta_df["age"], bins=[0, 20, 40, 60, 80, 120],
                              labels=["0-20", "21-40", "41-60", "61-80", "81+"])
full_df = results_df.merge(meta_df, on="patientId", how="left")

print("Sex:\n", meta_df["sex"].value_counts())
print("\nView position:\n", meta_df["view_position"].value_counts())
print("\nAge group:\n", meta_df["age_group"].value_counts().sort_index())

### Reflection Question 1
1. Is the dataset balanced by sex? By age group?
2. What problems could an imbalanced dataset cause when training a model?
3. Does this distribution reflect the real population of pneumonia patients? Why or why not?

*Your answer (double-click to edit):*

# 3. Overall performance
Before slicing by subgroup, the big picture — and why accuracy alone is misleading on imbalanced data.

In [ ]:
y_true, y_pred, y_prob = full_df["true_label"].values, full_df["predicted"].values, full_df["prob_pneumonia"].values
print("=" * 40)
print(f"  Accuracy:  {accuracy_score(y_true, y_pred):.3f}")
print(f"  Precision: {precision_score(y_true, y_pred, zero_division=0):.3f}")
print(f"  Recall:    {recall_score(y_true, y_pred, zero_division=0):.3f}")
print(f"  F1:        {f1_score(y_true, y_pred, zero_division=0):.3f}")
print(f"  ROC-AUC:   {roc_auc_score(y_true, y_prob):.3f}")
print("=" * 40)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=["Normal", "Pneumonia"]).plot(
    ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix", fontweight="bold")
fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[1].plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {roc_auc_score(y_true, y_prob):.3f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Random")
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve", fontweight="bold"); axes[1].legend(); axes[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

### Reflection Question 2
1. Which error is worse here: a **False Positive** (false alarm) or a **False Negative** (missed pneumonia)? Why?
2. Which matters more for a *screening* tool — Precision or Recall?
3. If you lowered the threshold from 0.5 to 0.3, what happens to each?

*Your answer:*

# 4. Bias by subgroup
Does the model perform equally well across **sex**, **age group**, and **view position** (AP = bedside, sicker patients; PA = standard upright)? Write the per-subgroup metric function — this is the heart of a bias audit.

In [ ]:
def subgroup_metrics(df, group_col):
    """Per-group accuracy / precision / recall / F1 / AUC for groups with >=10 patients."""
    rows = []
    for group, sub in df.groupby(group_col, observed=True):
        if len(sub) < 10:
            continue
        yt, yp, yprob = sub["true_label"].values, sub["predicted"].values, sub["prob_pneumonia"].values
        # AUC is undefined if a subgroup is all one class — guard it.
        # TODO: build one results row for this group:
        # - wrap roc_auc_score(yt, yprob) in try/except -> nan if single-class
        # - append a dict with: Group, N=len(sub), Prevalence (yt.mean()),
        # Accuracy, Recall, F1 (use sklearn, zero_division=0), ROC-AUC
        raise NotImplementedError("Build the per-group metrics row, then delete this line.")
    return pd.DataFrame(rows).set_index("Group")

sex_metrics  = subgroup_metrics(full_df[full_df["sex"].isin(["M", "F"])], "sex")
age_metrics  = subgroup_metrics(full_df.dropna(subset=["age_group"]), "age_group")
view_metrics = subgroup_metrics(full_df[full_df["view_position"].isin(["AP", "PA"])], "view_position")
print("By SEX\n", sex_metrics, "\n\nBy AGE\n", age_metrics, "\n\nBy VIEW\n", view_metrics)

In [ ]:
#@title ROC by subgroup { display-mode: "form" }
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (col, groups, title) in zip(axes, [
        ("sex", ["M", "F"], "ROC by Sex"),
        ("age_group", ["0-20", "21-40", "41-60", "61-80", "81+"], "ROC by Age"),
        ("view_position", ["AP", "PA"], "ROC by View")]):
    ax.plot([0, 1], [0, 1], "k--", alpha=.4)
    for grp in groups:
        sub = full_df[full_df[col] == grp]
        if len(sub) < 10 or sub["true_label"].nunique() < 2:
            continue
        fpr, tpr, _ = roc_curve(sub["true_label"], sub["prob_pneumonia"])
        ax.plot(fpr, tpr, lw=2, label=f"{grp} (AUC={roc_auc_score(sub['true_label'], sub['prob_pneumonia']):.2f})")
    ax.set_title(title, fontweight="bold"); ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(fontsize=9); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

### Reflection Question 3
1. Which subgroup does the model do **worst** on (focus on Recall — missing pneumonia is dangerous)?
2. Were any results surprising?
3. Why might performance differ on AP vs PA X-rays? *(AP = lying down, sicker patients; PA = standard upright.)*

*Your answer:*

# 5. Formalizing fairness — independence, separation, sufficiency

In Part 4 you compared recall by eye. The fairness literature gives three precise, *competing* definitions of "fair" — the same ones behind the **COMPAS** recidivism debate, where two sides each called the tool fair using a **different** definition and both were technically right.

Let $y$ = true label (1 = pneumonia), $d$ = decision (1 = flagged), $A$ = patient group.

- **Independence (demographic parity):** $P(d{=}1 \mid A)$ equal across groups — same flag rate for everyone. But prevalence really differs by group, so equal flag rates can *under-screen* a high-prevalence group.
- **Separation (equalized odds):** equal **TPR** and **FPR** across groups — among truly sick patients, the same catch rate; among healthy, the same false-alarm rate. (ProPublica's criterion.)
- **Sufficiency (calibration / equal PPV):** $P(y{=}1 \mid d{=}1, A)$ equal — a positive flag means the same thing for everyone. (Northpointe's criterion.)

**The impossibility:** when prevalence differs across groups, you cannot satisfy separation and sufficiency at once (except in trivial cases). You must *choose*.

In [ ]:
def fairness_table(df, group_col, groups):
    rows = []
    for g in groups:
        sub = df[df[group_col] == g]
        if len(sub) < 10:
            continue
        tn, fp, fn, tp = confusion_matrix(sub["true_label"], sub["predicted"], labels=[0, 1]).ravel()
        n, pos, neg, flagged = tn + fp + fn + tp, tp + fn, tn + fp, tp + fp
        rows.append({"Group": g, "N": n, "Prevalence": pos / n, "Selection P(d=1)": flagged / n,
                     "TPR (recall)": tp / pos if pos else float("nan"),
                     "FPR": fp / neg if neg else float("nan"),
                     "PPV (precision)": tp / flagged if flagged else float("nan")})
    return pd.DataFrame(rows).set_index("Group").round(3)

sex_fair  = fairness_table(full_df, "sex", ["M", "F"])
view_fair = fairness_table(full_df, "view_position", ["AP", "PA"])
print("Fairness by SEX\n", sex_fair, "\n\nFairness by VIEW\n", view_fair)

In [ ]:
def criterion_gaps(table):
    """Max-minus-min gap across groups for each fairness criterion (0 = perfectly equal)."""
    # TODO: return a pd.Series of max-min gaps across groups for:
    # Independence -> "Selection P(d=1)" column
    # Separation   -> "TPR (recall)" gap AND "FPR" gap
    # Sufficiency  -> "PPV (precision)" gap
    # each gap = table[col].max() - table[col].min()
    raise NotImplementedError("Compute the four criterion gaps, then delete this line.")

print("Gaps across SEX\n", criterion_gaps(sex_fair), "\n\nGaps across VIEW\n", criterion_gaps(view_fair))

### Reflection Question 4
1. Which criterion shows the largest gap across **view position**? Across **sex**?
2. Given the prevalence differences and the impossibility result, can this model satisfy separation *and* sufficiency at once? For a screening tool, which would you prioritize — and what does that cost?
3. If the same predicted probability maps to different real pneumonia rates for two groups, which criterion is violated?

*Your answer:*

# 6. Where does the model fail?
Look at the actual X-rays the model got wrong — especially the **false negatives** (missed pneumonia).

In [ ]:
false_negatives = full_df[(full_df["true_label"] == 1) & (full_df["predicted"] == 0)]
false_positives = full_df[(full_df["true_label"] == 0) & (full_df["predicted"] == 1)]
print(f"False negatives (missed): {len(false_negatives)} | False positives (false alarm): {len(false_positives)}")
print("\nFalse negatives by demographics:")
print(false_negatives[["sex", "age_group", "view_position"]].value_counts().head(8))

In [ ]:
#@title Sample misclassified X-rays { display-mode: "form" }
def show_errors(error_df, title, color, n=6):
    sample = error_df.sample(min(n, len(error_df)), random_state=42)
    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
        p = VAL_DIR / f"{row['patientId']}.jpg"
        if p.exists():
            ax.imshow(Image.open(p).convert("L"), cmap="bone")
        ax.set_title(f"{title}\nSex:{row.get('sex','?')} Age:{row.get('age','?')} View:{row.get('view_position','?')}\n"
                     f"P(pneu)={row['prob_pneumonia']:.2f}", fontsize=9, color=color, fontweight="bold")
        ax.axis("off")
    for ax in axes.ravel()[len(sample):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

if len(false_negatives): show_errors(false_negatives, "False Negative", "#d32f2f")
if len(false_positives): show_errors(false_positives, "False Positive", "#f57c00")

### Reflection Question 5
1. Do the false-negative X-rays look obviously wrong to *you*? What does that tell you about the task?
2. Is the model more confident on one sex than the other for true positives?
3. What would lowering the threshold to 0.3 do?

*Your answer:*

# 7. Looking inside — SHAP attribution maps

You inspected wrong images by eye; **SHAP** shows *which pixels* drove each prediction. It assigns every pixel a contribution toward (red) or against (blue) "pneumonia."

> **Why this matters for bias.** A model can be right for the wrong reason. If the red pixels land *outside* the lungs — borders, shoulders, text, tubes — the model may be exploiting a shortcut correlated with the label (scanner type, patient positioning) rather than reading pathology. That's one concrete mechanism behind an AP-vs-PA gap.

In [ ]:
if RUN_SHAP:
    import shap
    model.eval()
    background, _, _ = next(iter(DataLoader(val_dataset, batch_size=50, shuffle=True)))
    background = background.to(DEVICE)

    tp_ids = full_df[(full_df["true_label"] == 1) & (full_df["predicted"] == 1)]["patientId"].tolist()[:2]
    fn_ids = false_negatives["patientId"].tolist()[:2]
    pid_to_pos = {pid: i for i, pid in enumerate(val_dataset.df["patientId"])}
    pos = [pid_to_pos[p] for p in (tp_ids + fn_ids) if p in pid_to_pos]
    sample_imgs = torch.stack([val_dataset[i][0] for i in pos]).to(DEVICE)

    shap_values = shap.GradientExplainer(model, background).shap_values(sample_imgs)
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    disp = np.clip(np.transpose(sample_imgs.cpu().numpy(), (0, 2, 3, 1)) * std + mean, 0, 1)
    attr = shap_values[1] if isinstance(shap_values, list) else shap_values[..., 1]
    print("Red pixels push toward 'pneumonia', blue away.")
    shap.image_plot([np.transpose(attr, (0, 2, 3, 1))], disp)
else:
    print("RUN_SHAP is False — skipping attribution maps.")

### Reflection Question 6
1. For the cases the model got **right**, do the red pixels concentrate on the lung fields, or elsewhere?
2. For the **false negatives**, where is the model's attention?
3. Do these maps hint the model leans on view-related cues rather than lung pathology? What experiment would make that conclusive?

*Your answer:*

# 8. Can we fix it?

## 8.1 Data augmentation
Augmentation expands the training data with realistic variations. Preview some, then **design your own** pipeline.

In [ ]:
#@title Augmentation preview { display-mode: "form" }
sample_pid = results_df["patientId"].iloc[0]
orig = Image.open(VAL_DIR / f"{sample_pid}.jpg").convert("RGB")
augs = [("Original", orig), ("H-Flip", TF.hflip(orig)), ("Rotate +15", TF.rotate(orig, 15)),
        ("Bright +30%", TF.adjust_brightness(orig, 1.3)), ("Contrast +50%", TF.adjust_contrast(orig, 1.5)),
        ("Center Crop", TF.center_crop(orig, 180))]
fig, axes = plt.subplots(1, len(augs), figsize=(20, 4))
for ax, (name, im) in zip(axes, augs):
    ax.imshow(im, cmap="bone"); ax.set_title(name, fontsize=10, fontweight="bold"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Try it: design an augmentation pipeline. Add/remove transforms and justify them below.
# transforms.RandomRotation(degrees=10)
# transforms.ColorJitter(brightness=0.2, contrast=0.2)
# transforms.RandomAffine(degrees=0, translate=(0.05, 0.05))
# transforms.GaussianBlur(kernel_size=3)
my_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    # ── ADD YOUR AUGMENTATIONS BELOW ──
    transforms.RandomHorizontalFlip(),
    # ─────────────────────────────────
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
print(my_pipeline)

## 8.2 Per-group thresholds (equalized odds)
A practical fairness fix: use a **different decision threshold per subgroup** so each reaches a target recall. Implement the search.

In [ ]:
def best_threshold_for_recall(sub_df, target=0.80):
    """Highest threshold that still achieves >= target recall (fewest false alarms at that recall)."""
    best_t = 0.5
    # TODO: sweep thresholds t from 0.1 to 0.9 (step 0.01). For each, compute
    # recall at that threshold; keep the LARGEST t whose recall >= target.
    raise NotImplementedError("Implement the threshold sweep, then delete this line.")
    return best_t

TARGET = 0.80
print(f"Per-group threshold for {TARGET*100:.0f}% recall:\n")
for col, groups in [("sex", ["M", "F"]), ("view_position", ["AP", "PA"])]:
    for grp in groups:
        sub = full_df[full_df[col] == grp]
        if len(sub) < 10:
            continue
        t = best_threshold_for_recall(sub, TARGET)
        pnew = (sub["prob_pneumonia"] >= t).astype(int)
        print(f"{col}={grp}: threshold={t:.2f}  recall={recall_score(sub['true_label'], pnew, zero_division=0):.3f}  "
              f"accuracy={accuracy_score(sub['true_label'], pnew):.3f}")

### Reflection Question 7
1. After adjusting thresholds, do some groups end up with lower accuracy? Is that an acceptable trade-off?
2. Is it fair to use **different thresholds for different demographic groups**? Argue both sides.
3. Can you think of a sampling strategy (instead of augmentation) that might improve fairness?

*Your answer:*

# 9. Final reflection — the ethics of AI in medicine

Answer **at least 3 of 5** (2–3 sentences each):

**Q1.** Your model is 90% accurate overall but only 75% recall for elderly patients. A hospital wants to deploy it for everyone. Would you recommend it? What safeguards would you require?

**Q2.** A nonprofit wants to use it in rural clinics with older equipment. What are the risks, and what would you test first?

**Q3.** Who is responsible when an AI makes a wrong diagnosis — the doctor, the hospital, or the engineers? Does it matter whether the bias was known beforehand?

**Q4.** We had no race/ethnicity data, so we couldn't test for racial bias. Does absence of data mean absence of bias? How would you test without explicit labels?

**Q5.** Describe a scenario where *trying to fix a bias makes things worse*. (Real examples exist!)

*Your answer:*

---

### Wrap-up checklist
- [ ] Loaded/trained a model and generated validation predictions
- [ ] Computed overall + per-subgroup metrics, found the worst group
- [ ] Computed the formal fairness gaps (independence / separation / sufficiency)
- [ ] Inspected failures and SHAP maps
- [ ] Tried a fairness fix (augmentation and/or per-group thresholds)
- [ ] Answered the reflection questions

Great work — you've done a real model audit, the kind responsible medical-AI teams run before deployment.